# Experiment 9: HPO for compact + all three new features

This notebook performs AutoGluon hyperparameter tuning only for the all-feature variant from `exp_9_target_features.ipynb`:

- the 13 `compact` features;
- `monthly_target_lag`;
- `town_flat_type_target_mean`; and
- `nearest_mrt_name`.

It uses 50 random trials each for LightGBM and CatBoost. Hyperparameters are selected on the final 17 labelled months before the test period, matching the test forecast horizon, then refit for submission.

In [9]:
from pathlib import Path
import os
import sys

import numpy as np
import pandas as pd
from autogluon.tabular import TabularPredictor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

configured_root = os.environ.get('KDDM_PROJECT_ROOT')
candidate_roots = ([Path(configured_root)] if configured_root else []) + [
    Path.cwd(), *Path.cwd().parents, Path('/content/KDDM-project'),
    Path('/content/drive/MyDrive/KDDM-project'),
]
project_root = next((path for path in candidate_roots if (path / 'exploration/metul/preprocessing.py').exists()), None)
if project_root is None:
    raise FileNotFoundError('Set KDDM_PROJECT_ROOT to the project directory.')
experiment_dir = project_root / 'exploration' / 'metul'
sys.path.insert(0, str(experiment_dir))

from preprocessing import RentalPreprocessor, load_datasets
from exp_9_feature_engineering import (
    ALL_FEATURES, TARGET, TargetFeatureTransformer, labelled_history, make_model_frame,
    preprocess_partition, unlabelled_history,
)

RUN_ID = os.environ.get('AUTOGLUON_RUN_ID', pd.Timestamp.now().strftime('%Y%m%d_%H%M%S'))
data_dir, datasets = load_datasets(project_root / 'data')
raw_train = datasets['train'].copy()
raw_test = datasets['test'].copy()
raw_train['period'] = pd.to_datetime(raw_train['RENT_APPROVAL_DATE'] + '-01').dt.to_period('M')
raw_test['period'] = pd.to_datetime(raw_test['RENT_APPROVAL_DATE'] + '-01').dt.to_period('M')
test_periods = np.sort(raw_test['period'].unique())
LAG_MONTHS = len(test_periods)
assert LAG_MONTHS == 17, f'Expected 17 test months, found {LAG_MONTHS}'

# Use the last labelled 17-month window before the test period as a chronological holdout.
# This mirrors the test horizon and excludes overlapping March-2025 training rows.
TEST_ORIGIN = raw_test['period'].min()
periods = np.sort(raw_train['period'].unique())
eligible_validation_periods = periods[periods < TEST_ORIGIN]
VALIDATION_MONTHS = LAG_MONTHS
if len(eligible_validation_periods) < VALIDATION_MONTHS + 1:
    raise ValueError('Not enough labelled months before the test period for time-based validation.')
validation_periods = eligible_validation_periods[-VALIDATION_MONTHS:]
validation_cutoff = validation_periods[0]
raw_fit = raw_train.loc[raw_train['period'] < validation_cutoff].copy()
raw_valid = raw_train.loc[raw_train['period'].isin(validation_periods)].copy()
assert raw_fit['period'].max() < raw_valid['period'].min() < TEST_ORIGIN

validation_preprocessor = RentalPreprocessor(datasets)
fit_base = preprocess_partition(validation_preprocessor, raw_fit, fit=True)
valid_base = preprocess_partition(validation_preprocessor, raw_valid)
fit_history = labelled_history(raw_fit, fit_base)
valid_history = labelled_history(raw_valid, valid_base)
validation_target_features = TargetFeatureTransformer(LAG_MONTHS)
fit_target_features = validation_target_features.fit_transform_train(fit_history)
valid_target_features = validation_target_features.transform(valid_history)
fit_model = make_model_frame(fit_base, fit_target_features, datasets['mrt'])
valid_model = make_model_frame(valid_base, valid_target_features, datasets['mrt'])
y_fit = raw_fit[TARGET].astype(float).reset_index(drop=True)
y_valid = raw_valid[TARGET].astype(float).reset_index(drop=True)
assert set(ALL_FEATURES).issubset(fit_model.columns)
assert set(ALL_FEATURES).issubset(valid_model.columns)
print(
    f'Prepared {len(ALL_FEATURES)} features; time-based validation: '
    f'{validation_periods[0]} to {validation_periods[-1]} ({len(validation_periods)} months). '
    f'Fit rows: {len(raw_fit):,}; validation rows: {len(raw_valid):,}.'
)


[categorical_clean] HDB lookup | TOWN: 26 -> 26 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | BLOCK: 2735 -> 2735 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | STREET: 576 -> 576 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | POSTAL_CODE: 9180 -> 9180 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | SUBZONE: 156 -> 156 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | PLANNING_AREA: 31 -> 31 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces


In [10]:
from autogluon.common import space

HPO_TRIALS_PER_MODEL = int(os.environ.get('AUTOGLUON_HPO_TRIALS_PER_MODEL', '50'))
HPO_TIME_LIMIT_SECONDS = int(os.environ.get('AUTOGLUON_HPO_TIME_LIMIT', '1800'))
HPO_HYPERPARAMETERS = {
    'GBM': {
        'num_leaves': space.Int(16, 256),
        # LightGBM accepts num_iterations as its boosting-round parameter.
        'num_iterations': space.Int(100, 500),
        'learning_rate': space.Real(0.01, 0.20, log=True),
        'min_data_in_leaf': space.Int(10, 200),
        'feature_fraction': space.Real(0.6, 1.0),
        'bagging_fraction': space.Real(0.6, 1.0),
    },
    'CAT': {
        'depth': space.Int(4, 10),
        # CatBoost uses iterations for the same number-of-boosting-rounds setting.
        'iterations': space.Int(100, 500),
        'learning_rate': space.Real(0.01, 0.20, log=True),
        'l2_leaf_reg': space.Real(1.0, 20.0, log=True),
    },
}
HPO_KWARGS = {
    'num_trials': HPO_TRIALS_PER_MODEL,
    'scheduler': 'local',
    'searcher': 'random',
}

validation_predictor = TabularPredictor(
    label=TARGET, problem_type='regression', eval_metric='root_mean_squared_error',
    path=str(experiment_dir / 'ag_exp_9_hpo' / RUN_ID / 'validation_all_features'),
).fit(
    train_data=fit_model[ALL_FEATURES].assign(**{TARGET: y_fit.to_numpy()}),
    tuning_data=valid_model[ALL_FEATURES].assign(**{TARGET: y_valid.to_numpy()}),
    hyperparameters=HPO_HYPERPARAMETERS,
    hyperparameter_tune_kwargs=HPO_KWARGS,
    presets='medium_quality', time_limit=HPO_TIME_LIMIT_SECONDS, verbosity=2,
)
valid_predictions = validation_predictor.predict(valid_model[ALL_FEATURES]).astype(float).clip(lower=0)
validation_metrics = pd.DataFrame([{
    'feature_set': 'compact_plus_lag_mean_and_nearest_mrt_hpo',
    'feature_count': len(ALL_FEATURES),
    'RMSE': mean_squared_error(y_valid, valid_predictions) ** 0.5,
    'MAE': mean_absolute_error(y_valid, valid_predictions),
    'R2': r2_score(y_valid, valid_predictions),
}])
display(validation_metrics)
display(validation_predictor.leaderboard(
    valid_model[ALL_FEATURES].assign(**{TARGET: y_valid.to_numpy()}), silent=True
))
print(f'HPO trials requested: {HPO_TRIALS_PER_MODEL * len(HPO_HYPERPARAMETERS)} total ({HPO_TRIALS_PER_MODEL} per model family).')


Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.6.2
Python Version:     3.12.14
Operating System:   Darwin
Platform Machine:   arm64
Platform Version:   Darwin Kernel Version 25.6.0: Fri Jul 31 19:16:20 PDT 2026; root:xnu-12377.161.14~5/RELEASE_ARM64_T8142
CPU Count:          10
Pytorch Version:    2.10.0
CUDA Version:       CUDA is not available
GPU Count:          WARNING: Exception was raised when calculating GPU count (AssertionError)
Memory Avail:       3.72 GB / 16.00 GB (23.3%)
Disk Space Avail:   183.50 GB / 460.38 GB (39.9%)
Presets specified: ['medium_quality']
Beginning AutoGluon training ... Time limit = 1800s
AutoGluon will save models to "/Users/metul/Desktop/KDDM-project/exploration/metul/ag_exp_9_hpo/20260924_161140/validation_all_features"
Train Data Rows:    96865
Train Data Columns: 16
Tuning Data Rows:    50103
Tuning Data Columns: 16
Label Column:       MONTHLY_RENT
Problem Type:       regression
Preprocessi

KeyboardInterrupt: 

In [11]:
# Refit all learned preprocessing and target features on every labelled row.
final_preprocessor = RentalPreprocessor(datasets)
full_base = preprocess_partition(final_preprocessor, raw_train, fit=True)
test_base = preprocess_partition(final_preprocessor, raw_test)
full_history = labelled_history(raw_train, full_base)
test_history = unlabelled_history(raw_test, test_base)
final_target_features = TargetFeatureTransformer(LAG_MONTHS)
full_target_features = final_target_features.fit_transform_train(full_history)
test_target_features = final_target_features.transform(test_history)
full_model = make_model_frame(full_base, full_target_features, datasets['mrt'])
test_model = make_model_frame(test_base, test_target_features, datasets['mrt'])
y_full = raw_train[TARGET].astype(float).reset_index(drop=True)
assert set(ALL_FEATURES).issubset(full_model.columns)
assert set(ALL_FEATURES).issubset(test_model.columns)


[categorical_clean] HDB lookup | TOWN: 26 -> 26 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | BLOCK: 2735 -> 2735 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | STREET: 576 -> 576 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | POSTAL_CODE: 9180 -> 9180 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | SUBZONE: 156 -> 156 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | PLANNING_AREA: 31 -> 31 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces


In [12]:
# This mirrors the earlier HPO workflow: tune again while fitting the final full-data model.
final_predictor = TabularPredictor(
    label=TARGET, problem_type='regression', eval_metric='root_mean_squared_error',
    path=str(experiment_dir / 'ag_exp_9_hpo' / RUN_ID / 'final_all_features'),
).fit(
    train_data=full_model[ALL_FEATURES].assign(**{TARGET: y_full.to_numpy()}),
    hyperparameters=HPO_HYPERPARAMETERS,
    hyperparameter_tune_kwargs=HPO_KWARGS,
    presets='medium_quality', time_limit=HPO_TIME_LIMIT_SECONDS, verbosity=2,
)
submission = pd.DataFrame({
    'Id': raw_test.index,
    'Predicted': final_predictor.predict(test_model[ALL_FEATURES]).astype(float).clip(lower=0),
})
submission_path = experiment_dir / 'exp_9_hpo_lag_mean_nearest_mrt_submission.csv'
submission.to_csv(submission_path, index=False)
print(f'Saved {len(submission):,} predictions to {submission_path}')
display(submission.head())


Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.6.2
Python Version:     3.12.14
Operating System:   Darwin
Platform Machine:   arm64
Platform Version:   Darwin Kernel Version 25.6.0: Fri Jul 31 19:16:20 PDT 2026; root:xnu-12377.161.14~5/RELEASE_ARM64_T8142
CPU Count:          10
Pytorch Version:    2.10.0
CUDA Version:       CUDA is not available
GPU Count:          WARNING: Exception was raised when calculating GPU count (AssertionError)
Memory Avail:       3.79 GB / 16.00 GB (23.7%)
Disk Space Avail:   183.47 GB / 460.38 GB (39.9%)
Presets specified: ['medium_quality']
Beginning AutoGluon training ... Time limit = 1800s
AutoGluon will save models to "/Users/metul/Desktop/KDDM-project/exploration/metul/ag_exp_9_hpo/20260924_161140/final_all_features"
Train Data Rows:    150000
Train Data Columns: 16
Label Column:       MONTHLY_RENT
Problem Type:       regression
Preprocessing data...
Using Feature Generators to preprocess the d

Saved 50,000 predictions to /Users/metul/Desktop/KDDM-project/exploration/metul/exp_9_hpo_lag_mean_nearest_mrt_submission.csv


,Id,Predicted
0,0,2741.998291
1,1,3619.083984
2,2,3249.334229
3,3,3166.235840
4,4,3268.903809
